# Open-World Amodal Appearance Completion (CVPR 2025) - Reproduce

Notebook này dùng để cài đặt môi trường và chạy thử nghiệm pipeline gốc của tác giả trên **Google Colab**.

### ⚠️ Yêu cầu quan trọng về phần cứng:
- Bạn bắt buộc phải chọn GPU **A100** trong Runtime để tránh bị tràn bộ nhớ (Out Of Memory) với mô hình LISA-13B.
- Cách cấu hình: **Runtime** -> **Change runtime type** -> Chọn **GPU** -> Chọn **A100**.

## Bước 1: Cài đặt Miniconda (Chạy môi trường Python 3.10)
Do code của tác giả được thiết kế chặt chẽ và phụ thuộc nhiều vào các thư viện chạy trên Python 3.10.

In [ ]:
import os
if not os.path.exists('/opt/conda/bin/python'):
    print('Đang tải và cài đặt Miniconda (Python 3.10)...')
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-py310_23.11.0-2-Linux-x86_64.sh
    !bash Miniconda3-py310_23.11.0-2-Linux-x86_64.sh -b -p /opt/conda 2>/dev/null
    !rm Miniconda3-py310_23.11.0-2-Linux-x86_64.sh
    print('Cài đặt Miniconda hoàn tất!')
else:
    print('Miniconda đã có sẵn trong session này.')

!/opt/conda/bin/python --version

## Bước 2: Clone Repository cá nhân của bạn
Hãy thay thế giá trị biến `GITHUB_USERNAME` bằng tên người dùng GitHub của bạn trước khi chạy.

In [ ]:
import os

# ⚠️ THAY TÊN GITHUB CỦA BẠN VÀO ĐÂY
GITHUB_USERNAME = "your_github_username"

if not os.path.exists('/content/amodal'):
    !git clone https://github.com/{GITHUB_USERNAME}/OWAmodal_Reproduce.git /content/amodal

%cd /content/amodal

## Bước 3: Cài đặt Dependencies chính
Chúng ta sẽ cài đặt PyTorch phiên bản 1.13.1 đúng với CUDA 11.7, sau đó cài đặt các thư viện trong `requirements.txt`.

In [ ]:
# 1. Cài đặt PyTorch 1.13.1 và Torchvision tương thích
!/opt/conda/bin/pip install torch==1.13.1+cu117 torchvision==0.14.1+cu117 -f https://download.pytorch.org/whl/torch_stable.html

# 2. Cài đặt các thư viện từ requirements.txt
!/opt/conda/bin/pip install -r requirements.txt

# 3. Cài đặt các thư viện bổ trợ cho LISA
!/opt/conda/bin/pip install einops bitsandbytes bleach sentencepiece protobuf

# 4. Cài đặt DeepSpeed 0.11.2 (cần hạ cấp setuptools trước để tránh lỗi setup.py)
!/opt/conda/bin/pip install "setuptools<70" "wheel<0.42"
!DS_BUILD_OPS=0 /opt/conda/bin/pip install --no-build-isolation deepspeed==0.11.2

## Bước 4: Tải các Sub-repositories và Checkpoints (~30 GB)
Bước này tải các model hỗ trợ như Grounded-SAM, RAM++, InstaOrder để phân tích occlusion.

In [ ]:
import os

# --- Grounded-Segment-Anything (SAM + GroundingDINO) ---
if not os.path.exists('Grounded-Segment-Anything'):
    print('Đang cài đặt Grounded-Segment-Anything...')
    !git clone https://github.com/IDEA-Research/Grounded-Segment-Anything
    !cd Grounded-Segment-Anything && /opt/conda/bin/pip install -e ./segment_anything
    !cd Grounded-Segment-Anything && BUILD_WITH_CUDA=0 /opt/conda/bin/pip install --no-build-isolation -e ./GroundingDINO
    !wget -q -P Grounded-Segment-Anything https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
    !wget -q -P Grounded-Segment-Anything https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth

# --- RAM++ (Tự động phát hiện nhãn ảnh) ---
if not os.path.exists('recognize-anything'):
    print('Đang cài đặt RAM++...')
    !git clone https://github.com/xinyu1205/recognize-anything.git
    !cd recognize-anything && /opt/conda/bin/pip install .
    !wget -q -P recognize-anything https://huggingface.co/xinyu1205/recognize-anything-plus-model/resolve/main/ram_plus_swin_large_14m.pth

# --- InstaOrder (Phân tích thứ tự occlusion) ---
if not os.path.exists('InstaOrder'):
    print('Đang cài đặt InstaOrder...')
    !git clone https://github.com/POSTECH-CVLab/InstaOrder
    !mkdir -p InstaOrder/InstaOrder_ckpt
    !/opt/conda/bin/pip install -q gdown
    !gdown --id 1_GEmCmofLSkJZnidfp4vsQb2Nqq5aqBU -O InstaOrder_ckpt.zip
    !unzip -q InstaOrder_ckpt.zip
    !mv InstaOrder_ckpt/InstaOrder_InstaOrderNet_od.pth.tar InstaOrder/InstaOrder_ckpt/
    !rm -rf InstaOrder_ckpt.zip InstaOrder_ckpt

# --- LISA (Phân tách ngữ nghĩa) ---
if not os.path.exists('LISA'):
    print('Đang cấu hình LISA...')
    !git clone https://github.com/dvlab-research/LISA.git
    !wget -q -O LISA/app.py https://raw.githubusercontent.com/saraao/amodal/main/LISA/app.py
    !sed -i 's|"./LISA-13B-llama2-v1"|"xinlai/LISA-13B-llama2-v1"|g' LISA/app.py
    !sed -i 's/AutoConfig.register("llava", LlavaConfig)/AutoConfig.register("llava", LlavaConfig, exist_ok=True)/' LISA/model/llava/model/language_model/llava_llama.py
    !sed -i 's/from .language_model.llava_mpt/# from .language_model.llava_mpt/' LISA/model/llava/model/__init__.py

print('=== Tất cả sub-repos và checkpoints đã sẵn sàng ===')

## Bước 5: Tải trọng số LISA Model (~26 GB)
Quá trình này sẽ mất từ 10 - 20 phút tùy thuộc vào tốc độ mạng của Colab.

In [ ]:
!/opt/conda/bin/huggingface-cli download xinlai/LISA-13B-llama2-v1 --resume-download

## Bước 6: Khởi chạy LISA Server (Chạy ngầm)
Khởi động Server phân khúc ngôn ngữ LISA bằng chế độ `precision bf16` để tối ưu hóa hiệu năng trên A100.

In [ ]:
import subprocess, time, os, signal

# Kill server LISA cũ nếu có
result = subprocess.run('pgrep -f app.py', shell=True, capture_output=True, text=True)
for pid in result.stdout.strip().split('\n'):
    if pid:
        try:
            os.kill(int(pid), signal.SIGKILL)
            print(f'Đã dừng tiến trình LISA cũ: PID {pid}')
        except ProcessLookupError:
            pass
time.sleep(3)

# Chạy server LISA ngầm, lưu log vào file lisa.log
process = subprocess.Popen(
    'cd /content/amodal/LISA && /opt/conda/bin/python app.py --version xinlai/LISA-13B-llama2-v1 --precision bf16',
    shell=True,
    stdout=open('/content/amodal/lisa.log', 'w'),
    stderr=subprocess.STDOUT
)
print(f'LISA Server đang khởi chạy với PID: {process.pid} (Loading model ~90s)...')

## Bước 7: Kiểm tra trạng thái của LISA Server
Chúng ta sẽ lặp lại kiểm tra xem file log đã thông báo Server sẵn sàng chưa.

In [ ]:
import time
print('Đang kiểm tra trạng thái khởi động của server...')
for i in range(18):
    time.sleep(10)
    if os.path.exists('/content/amodal/lisa.log'):
        with open('/content/amodal/lisa.log', 'r') as f:
            log = f.read()
        if 'Running on local URL' in log:
            print('\n✅ LISA Server đã sẵn sàng và hoạt động!')
            break
    print(f'  Chờ đợi... {(i+1)*10}s')
else:
    print('\n⚠️ Quá thời gian chờ (Timeout) - Hãy kiểm tra file lisa.log dưới đây:')

if os.path.exists('/content/amodal/lisa.log'):
    !tail -n 15 /content/amodal/lisa.log

## Bước 8: Sửa đổi các đường dẫn cứng (Patch Paths) trong `main.py`
Mã nguồn của tác giả mặc định để các đường dẫn placeholder. Đoạn mã dưới đây tự động sửa chúng thành đường dẫn chạy trên Colab.

In [ ]:
with open('/content/amodal/main.py', 'r') as f:
    code = f.read()

code = code.replace(
    'PROJECT_PATH = "/your/path/here/"',
    'PROJECT_PATH = "/content/"'
).replace(
    'LISA_OUTPUT_PATH = "/your/path/here/LISAoutput/"',
    'LISA_OUTPUT_PATH = "/content/amodal/LISAoutput/"'
)

with open('/content/amodal/main.py', 'w') as f:
    f.write(code)

import os
os.makedirs('/content/amodal/LISAoutput/', exist_ok=True)
print('✅ Đã sửa đổi cấu hình đường dẫn thành công!')

## Bước 9: Chạy kiểm thử trên ảnh ví dụ của tác giả
Pipeline sẽ xử lý các ảnh mẫu có sẵn trong thư mục `images_example/` và ghi kết quả vào thư mục `output_example/`.

In [ ]:
%cd /content/amodal
!/opt/conda/bin/python main.py \
    --input_dir ./images_example/ \
    --img_filenames_txt ./img_filenames_example.txt \
    --json_label_path ./example_annotation.json \
    --output_dir ./output_example \
    --line_num 0

## Bước 10: Trực quan hóa kết quả
Hiển thị các ảnh đã hoàn thiện (Appearance Completion) và mặt nạ đầy đủ của vật thể (Amodal Mask).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import glob, os

# Tìm kiếm các file output
completions = sorted(glob.glob('/content/amodal/output_example/**/amodal_completions_raw/*.jpg', recursive=True))
segmentations = sorted(glob.glob('/content/amodal/output_example/**/amodal_segmentation/*.png', recursive=True))

n = min(len(completions), 5)
if n == 0:
    print('Không tìm thấy ảnh kết quả. Hãy kiểm tra lại log chạy ở Bước 9.')
else:
    fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
    if n == 1:
        axes = axes.reshape(2, 1)
    for i in range(n):
        axes[0][i].imshow(Image.open(completions[i]))
        axes[0][i].set_title(os.path.basename(completions[i]), fontsize=8)
        axes[0][i].axis('off')
        if i < len(segmentations):
            axes[1][i].imshow(Image.open(segmentations[i]))
            axes[1][i].axis('off')
    axes[0][0].set_ylabel('Appearance Completion', fontsize=12)
    axes[1][0].set_ylabel('Amodal Segmentation Mask', fontsize=12)
    plt.tight_layout()
    plt.show()